# RAG Misinformation Experiments — Results

This notebook loads `results/experiment_results.csv` (produced by
`python -m src.evaluator`) and draws simple charts comparing the three
experiments.

- **Exp 1**: real docs only (baseline)
- **Exp 2**: real + poisoned, no verification
- **Exp 3**: real + poisoned, with verification

We expect the **hallucination rate** to rise from Exp 1 → Exp 2, then fall
again in Exp 3 if the verification prompt helps.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Make sure we can find the results file whether the notebook is run from
# the repo root or from inside notebooks/.
CSV_PATH = "../results/experiment_results.csv"
if not os.path.exists(CSV_PATH):
    CSV_PATH = "results/experiment_results.csv"

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows from {CSV_PATH}")
df.head()

## Aggregate metrics per experiment

In [ ]:
# accuracy = mean of is_correct over all questions
# hallucination_rate = mean of is_hallucinated over poisoned-target questions only
order = ["exp1_baseline", "exp2_poisoned", "exp3_poisoned_verify"]

accuracy = df.groupby("experiment")["is_correct"].mean()
poisoned = df[df["is_poisoned_target"] == True]
hallucination = poisoned.groupby("experiment")["is_hallucinated"].mean()

summary = pd.DataFrame({
    "accuracy": accuracy,
    "hallucination_rate": hallucination,
}).reindex(order)
summary

## Chart: hallucination rate per experiment

In [ ]:
labels = ["Exp 1\n(baseline)", "Exp 2\n(poisoned)", "Exp 3\n(poisoned+verify)"]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, summary["hallucination_rate"], color=["#4c72b0", "#c44e52", "#55a868"])
ax.set_ylabel("Hallucination rate")
ax.set_ylim(0, 1)
ax.set_title("How often the model repeated the poisoned (fake) answer")
for i, v in enumerate(summary["hallucination_rate"]):
    ax.text(i, v + 0.02, f"{v:.0%}", ha="center")
plt.tight_layout()
plt.show()

## Chart: accuracy vs hallucination side by side

In [ ]:
import numpy as np

x = np.arange(len(order))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width/2, summary["accuracy"], width, label="accuracy", color="#55a868")
ax.bar(x + width/2, summary["hallucination_rate"], width, label="hallucination_rate", color="#c44e52")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1)
ax.set_ylabel("rate")
ax.set_title("Accuracy vs Hallucination rate by experiment")
ax.legend()
plt.tight_layout()
plt.show()

## Inspect individual answers

Useful for the report — see exactly what the model said in each condition.

In [ ]:
cols = ["experiment", "question", "model_answer", "is_correct", "is_hallucinated", "poisoned_doc_retrieved"]
pd.set_option("display.max_colwidth", 80)
df[cols]